# Advanced Feature Engineering & Model Optimization

This notebook covers:
1. **Advanced Feature Engineering**
2. **Feature Scaling & Selection**
3. **Hyperparameter Tuning**
4. **Ensemble Methods**
5. **Final Model Evaluation**

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries imported successfully!")

---
## Load Data & Initial Preprocessing

In [ ]:
df = pd.read_csv('train.csv')

# Preprocessing
df['Age'].fillna(df['Age'].median(), inplace=True)
df.drop(columns=['Cabin'], inplace=True, errors='ignore')
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=False)

print(f"Dataset shape: {df.shape}")
print(f"\nDataset info:\n{df.info()}")

---
## Advanced Feature Engineering

In [ ]:
# 1. Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
title_counts = df['Title'].value_counts()
rare_titles = title_counts[title_counts < 5].index
df['Title'] = df['Title'].apply(lambda x: 'Rare' if x in rare_titles else x)
df['Title'] = df['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

print(f"✓ Title feature engineered")
print(f"  Title values: {sorted(df['Title'].dropna().unique())}")

# 2. Family Size Feature
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print(f"✓ Family size features created")

# 3. Fare per person
df['Fare_per_Person'] = df['Fare'] / df['FamilySize']

print(f"✓ Fare per person calculated")

# 4. Age groups
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 5, 12, 18, 35, 60, 120], 
                         labels=['Baby', 'Child', 'Teen', 'Adult', 'Senior', 'Elderly'])
df['Age_Group'] = pd.factorize(df['Age_Group'])[0]

print(f"✓ Age groups created")

# 5. High Fare indicator
df['High_Fare'] = (df['Fare'] > df['Fare'].quantile(0.75)).astype(int)

print(f"✓ High fare indicator added")

print(f"\nEnhanced dataset shape: {df.shape}")

---
## Feature Selection

In [ ]:
# Prepare features and target
drop_cols = ['Survived', 'Name', 'Ticket', 'PassengerId']
X = df.drop(columns=drop_cols)
y = df['Survived']

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Feature selection: Select top K features
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)
selected_features = X_scaled.columns[selector.get_support()].tolist()

print("Top 10 Selected Features (by F-score):")
feature_scores = pd.DataFrame({
    'Feature': X_scaled.columns,
    'Score': selector.scores_
}).sort_values('Score', ascending=False)

print(feature_scores.to_string(index=False))
print(f"\nSelected features: {selected_features}")

---
## Train-Test Split

In [ ]:
# Split using selected features
X_selected_df = X_scaled[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_selected_df, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size  : {X_train.shape}")
print(f"Test set size      : {X_test.shape}")
print(f"\nClass distribution (Training):")
print(y_train.value_counts())
print(f"\nClass distribution (Test):")
print(y_test.value_counts())

---
## Hyperparameter Tuning

### GridSearchCV for Random Forest

In [ ]:
# Random Forest Grid Search
rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print("Tuning Random Forest...")
rf_grid.fit(X_train, y_train)

print(f"\nBest RF parameters: {rf_grid.best_params_}")
print(f"Best RF ROC-AUC: {rf_grid.best_score_:.4f}")

rf_best = rf_grid.best_estimator_

### GridSearchCV for Gradient Boosting

In [ ]:
# Gradient Boosting Grid Search
gb_params = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0]
}

gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

print("Tuning Gradient Boosting...")
gb_grid.fit(X_train, y_train)

print(f"\nBest GB parameters: {gb_grid.best_params_}")
print(f"Best GB ROC-AUC: {gb_grid.best_score_:.4f}")

gb_best = gb_grid.best_estimator_

---
## Ensemble Voting Classifier

In [ ]:
# Logistic Regression baseline
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

# Create Voting Classifier
voting_clf = VotingClassifier(
    estimators=[('lr', lr), ('rf', rf_best), ('gb', gb_best)],
    voting='soft'
)

print("Training Voting Ensemble...")
voting_clf.fit(X_train, y_train)
print("✓ Voting ensemble trained")

---
## Final Model Evaluation

In [ ]:
# Evaluate all models
models = {
    'Logistic Regression': lr,
    'Random Forest (Tuned)': rf_best,
    'Gradient Boosting (Tuned)': gb_best,
    'Voting Ensemble': voting_clf
}

results = []

print("\n" + "="*90)
print("FINAL MODEL EVALUATION ON TEST SET")
print("="*90)

for model_name, model in models.items():
    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    })
    
    print(f"\n{model_name}")
    print("-" * 50)
    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print("\n" + "="*90)
print("MODEL RANKING (by ROC-AUC)")
print("="*90)
print(results_df.to_string(index=False))
print("\n" + "="*90)

---
## Cross-Validation Analysis

In [ ]:
# 5-Fold Cross-validation for all models
print("\n5-Fold Cross-Validation Results:")
print("="*70)

cv_results = {}
for model_name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    cv_results[model_name] = cv_scores
    
    print(f"\n{model_name}")
    print(f"  Fold scores: {[f'{s:.4f}' for s in cv_scores]}")
    print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\n" + "="*70)

---
## Feature Importance from Best Model

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'Feature': selected_features,
    'Importance': rf_best.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance (Random Forest - Tuned):")
print(feature_importance.to_string(index=False))

# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature', palette='viridis')
plt.title('Top Feature Importance (Tuned Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('final_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print("\nSaved: final_feature_importance.png")

---
## Model Comparison Visualization

In [ ]:
# Model comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    results_df.plot(x='Model', y=metric, kind='bar', ax=ax, legend=False, color='steelblue')
    ax.set_title(f'{metric} Comparison')
    ax.set_ylabel(metric)
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

---
## Final Recommendations

In [ ]:
print("\n" + "="*80)
print("FINAL ANALYSIS SUMMARY & RECOMMENDATIONS")
print("="*80)

best_model_name = results_df.iloc[0]['Model']
best_model_auc = results_df.iloc[0]['ROC-AUC']

print(f"\n✓ BEST PERFORMING MODEL: {best_model_name}")
print(f"  - ROC-AUC Score: {best_model_auc:.4f}")
print(f"  - Test Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")
print(f"  - F1-Score: {results_df.iloc[0]['F1-Score']:.4f}")

print("\n✓ KEY FEATURES (Top 5):")
for idx, row in feature_importance.head(5).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

print("\n✓ IMPROVEMENTS MADE:")
print("  - Advanced feature engineering (Title, Family Size, Fare per person)")
print("  - Feature selection reducing dimensionality")
print("  - Hyperparameter tuning using GridSearchCV")
print("  - Ensemble voting classifier for robustness")
print("  - Cross-validation for generalization assessment")

print("\n✓ PRODUCTION RECOMMENDATIONS:")
print(f"  - Deploy {best_model_name} for inference")
print("  - Monitor model drift with new data")
print("  - Use ensemble for critical predictions")
print("  - Set confidence threshold > 0.65 for high-precision decisions")

print("\n" + "="*80)